### Libraries imports

In [50]:
import re
import os
import faiss
import numpy as np
import pickle

from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv

/home/mikemadeira/anaconda3/envs/rag-aws-loka-env/lib/python3.9/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


### Data Pre-processing

In [115]:
def get_all_files_in_directory(directory_path, file_extension=".md"):
    """Retrieves all files with a specific extension from the directory."""
    files = []
    for root, dirs, file_names in os.walk(directory_path):
        for file_name in file_names:
            if file_name.endswith(file_extension):
                files.append(os.path.join(root, file_name))
    return files

In [116]:
def clean_text(text):
    """Cleans the input text by removing unnecessary characters."""
    # Remove extra spaces and newlines
    text = re.sub(r'\s+', ' ', text)
    
    # You can add additional cleaning steps here (e.g., removing special symbols)
    return text.strip()

In [117]:
def read_and_chunk_file(file_path, chunk_size=200, overlap=50):
    """Reads and chunks a single file."""
    with open(file_path, 'r', encoding='utf-8') as file:
        raw_text = file.read()
    
    # Clean the text
    cleaned_text = clean_text(raw_text)
    
    # Chunk the text
    words = cleaned_text.split()
    chunks = []
    i = 0
    
    while i < len(words):
        chunk = words[i:i + chunk_size]
        # print(f"Words from {i} to {i + chunk_size}:\n{chunk}")
        chunks.append(' '.join(chunk))  # Return only the chunk
        i += chunk_size - overlap
    
    return chunks

In [118]:
def parse_and_chunk_from_directory(directory_path, chunk_size=200, overlap=50, file_extension=".md"):
    """Parses and chunks all documents in the directory."""
    files = get_all_files_in_directory(directory_path, file_extension)
    
    all_chunks = []
    
    for file_path in files:
        file_chunks = read_and_chunk_file(file_path, chunk_size, overlap)
        all_chunks.extend(file_chunks)  # Add all chunks from this file to the overall list
    
    return all_chunks

In [119]:
# Specify the directory containing the markdown files
directory_path = '../data/sagemaker_documentation'  # Change this to your directory path
# directory_path = '../data/test_folder'  # Change this to your directory path

# Parse and chunk the documents in the directory, with file names included
chunks = parse_and_chunk_from_directory(directory_path, chunk_size=200, overlap=50, file_extension=".md")

In [120]:
# Print the chunks along with the file names for debugging
for i, chunk in enumerate(chunks[:5]):  # Limiting to first 5 for demo
    print(f"Chunk {i+1}:")
    print(chunk)
    print("="*40)

Chunk 1:
# AWS::SageMaker::DataQualityJobDefinition DataQualityAppSpecification<a name="aws-properties-sagemaker-dataqualityjobdefinition-dataqualityappspecification"></a> Information about the container that a data quality monitoring job runs\. ## Syntax<a name="aws-properties-sagemaker-dataqualityjobdefinition-dataqualityappspecification-syntax"></a> To declare this entity in your AWS CloudFormation template, use the following syntax: ### JSON<a name="aws-properties-sagemaker-dataqualityjobdefinition-dataqualityappspecification-syntax.json"></a> ``` { "[ContainerArguments](#cfn-sagemaker-dataqualityjobdefinition-dataqualityappspecification-containerarguments)" : [ String, ... ], "[ContainerEntrypoint](#cfn-sagemaker-dataqualityjobdefinition-dataqualityappspecification-containerentrypoint)" : [ String, ... ], "[Environment](#cfn-sagemaker-dataqualityjobdefinition-dataqualityappspecification-environment)" : {Key : Value, ...}, "[ImageUri](#cfn-sagemaker-dataqualityjobdefinition-dataqua

#### Generate Embeddings for Chunks

In [121]:
def generate_embeddings(chunks):
    """Generates embeddings for each chunk using a pre-trained sentence transformer."""
    model = SentenceTransformer('all-MiniLM-L6-v2')  # You can choose a different model if desired
    # Generate embeddings for each chunk
    embeddings = model.encode(chunks, convert_to_tensor=True)
    
    return embeddings

In [122]:
embeddings = generate_embeddings(chunks)

/home/mikemadeira/anaconda3/envs/rag-aws-loka-env/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [124]:
embeddings

tensor([[-0.0363, -0.0222,  0.0078,  ..., -0.0563, -0.0070, -0.0157],
        [-0.0388, -0.0363, -0.0076,  ..., -0.0738,  0.0423, -0.0503],
        [-0.0675, -0.0425, -0.0257,  ..., -0.0485, -0.0058, -0.0687],
        ...,
        [-0.0378, -0.0352,  0.0086,  ..., -0.0680, -0.0289,  0.0028],
        [-0.0129, -0.0189,  0.0027,  ..., -0.0867,  0.0011, -0.0122],
        [ 0.0354, -0.0780,  0.0411,  ..., -0.0147, -0.0439, -0.0083]])

In [125]:
embeddings.shape

torch.Size([911, 384])

### Data Storage

In [126]:
def create_faiss_index(embeddings):
    """Creates a FAISS index from the embeddings."""
    # Convert embeddings to a numpy array
    embeddings_np = np.array(embeddings)
    
    # Initialize FAISS index
    dimension = embeddings_np.shape[1]  # Embedding dimension
    index = faiss.IndexFlatL2(dimension)  # L2 distance for similarity search
    
    # Add embeddings to the index
    index.add(embeddings_np)
    
    return index

In [127]:
faiss_index = create_faiss_index(embeddings)

In [128]:
faiss_index

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7f48f7d5a480> >

### Query Retrieval

In [129]:
def search_query(query, model, index, chunks, top_k=5):
    """Searches for the most relevant chunks based on the input query."""
    # Embed the query
    query_embedding = model.encode([query], convert_to_tensor=False)
    
    # Search in the FAISS index for the most similar chunks
    distances, indices = index.search(np.array(query_embedding), top_k)
    
    # Retrieve the top-k chunks
    results = []
    for idx in indices[0]:
        results.append(chunks[idx])  # Get the chunk text
    
    return results

In [130]:
model = SentenceTransformer('all-MiniLM-L6-v2')  # The same model used for embeddings
user_query = "What is SageMaker?"  # Example query
results = search_query(user_query, model, faiss_index, chunks)

# Print the search results
for i, chunk in enumerate(results):
    print(f"Result {i+1}:")
    print(chunk)
    print("="*40)

/home/mikemadeira/anaconda3/envs/rag-aws-loka-env/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Result 1:
with SageMaker, see [Using RL with the SageMaker Python SDK](https://sagemaker.readthedocs.io/en/stable/using_rl.html)\.
Result 2:
SageMaker Neo\. For more information, see [Tutorial: Get started with SageMaker Edge Manager](get-started-with-edge-manager-on-greengrass.md)\.
Result 3:
examples repository](https://github.com/awslabs/amazon-sagemaker-examples/tree/master/reinforcement_learning) show how to build a custom container to use with SageMaker RL: ## Use Commercial Environments<a name="sagemaker-rl-environments-commercial"></a> You can use commercial environments, such as MATLAB and Simulink, in SageMaker RL by building your own container\. You need to manage your own licenses\.
Result 4:
# SageMaker Edge Manager<a name="sagemaker-edge-manager-component"></a> The Amazon SageMaker Edge Manager component \(`aws.greengrass.SageMakerEdgeManager`\) installs the SageMaker Edge Manager agent binary\. SageMaker Edge Manager provides model management for edge devices so you can 

### Indexes and chunks storage

In [131]:
def save_faiss_index(index, index_path):
    """Save the FAISS index to a file."""
    faiss.write_index(index, index_path)

def load_faiss_index(index_path):
    """Load the FAISS index from a file."""
    return faiss.read_index(index_path)

In [132]:
# Save FAISS index
faiss_index_path = 'faiss_index_file.index'
save_faiss_index(faiss_index, faiss_index_path)

In [133]:
# Load FAISS index
faiss_index_path = 'faiss_index_file.index'
faiss_index = load_faiss_index(faiss_index_path)

In [134]:
def save_chunks(chunks, chunks_path):
    """Save the list of chunks to a file using pickle."""
    with open(chunks_path, 'wb') as f:
        pickle.dump(chunks, f)

def load_chunks(chunks_path):
    """Load the list of chunks from a file using pickle."""
    with open(chunks_path, 'rb') as f:
        return pickle.load(f)

In [135]:
# Save chunks
chunks_path = 'chunks_file.pkl'
save_chunks(chunks, chunks_path)

In [136]:
# Load chunks
chunks_path = 'chunks_file.pkl'
chunks = load_chunks(chunks_path)

### LLM-based Q&A System

In [108]:
# Load environment variables from .env file
load_dotenv()

# Set your OpenAI API key
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
client = OpenAI()

In [113]:
def generate_answer(query, retrieved_chunks):
    """Generate an answer using an LLM based on the query and retrieved chunks."""
    context = "\n".join(retrieved_chunks)  # Combine all chunks into a single context
    
    # Create the prompt for the LLM
    prompt = f"Answer the following question based on the provided context:\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    # Send the prompt to the LLM (using OpenAI's GPT)
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": 'system', "content": 'You are an helpful assistant.'}, 
                  {"role": 'user', "content": prompt}],
    )
    
    print(f"Response from the LLM:\n{response}")
    # Extract the answer from the response
    answer = response.choices[0].message.content
    
    return answer

In [114]:
# Generate an answer using the retrieved chunks
if results:
    answer = generate_answer(user_query, results)
    print(f"Question: {user_query}")
    print(f"Answer: {answer}")

Response from the LLM:
ChatCompletion(id='chatcmpl-A9XOGon0ko2xMqcrjvPShHNIVfJsf', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="To create a model in Amazon SageMaker, follow these steps:\n\n### Step 1: Open the SageMaker Console\n1. Go to the [Amazon SageMaker console](https://us-east-2.console.aws.amazon.com/sagemaker/home).\n2. Ensure you are in the correct AWS Region for publishing your model.\n\n### Step 2: Navigate to Model Packages\n1. In the left navigation menu, select **Model packages**.\n2. Click **Create model package** to start creating your model package.\n\n### Step 3: Set Inference Specifications\n1. Provide a **Name** for your model package (e.g., *my-model-package*).\n2. Enter the **Location of inference image** by providing the URI of the inference image stored in Amazon Elastic Container Registry (Amazon ECR).\n3. (Optional) Provide any model artifacts stored in Amazon S3.\n\n### Step 4: Set Validation Specifica